# Crypto.Net · 09 — Praktik keamanan

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./09-security.en.ipynb)

Yang akan Anda lakukan: menelusuri kesalahan yang membuat dana hilang, dan melihat bagaimana Crypto.Net membantu menghindari masing-masing. Topiknya: rahasia di memori, keystore, jumlah yang presisi, alamat dengan checksum, perlindungan replay, verifikasi sebelum percaya, decoding input berbahaya, dan menjaga API key tetap di luar log.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.2.0"

In [ ]:
using System.Numerics;
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Evm;
using Crypto.Net.Bitcoin;
using Crypto.Net.Extensions;
"ready"

## 1. Rahasia disimpan di `SecureBuffer`, bukan `string`

`SecureBuffer` di-pin (GC tidak pernah menyalinnya) dan dinolkan saat `Dispose`. String tidak bisa diubah, sehingga tertinggal di memori dan muncul di crash dump. Aturan analyzer CNET002 menandai field dan property `string` yang tampak seperti rahasia.

In [ ]:
var secret = SecureBuffer.FromHex("4c0883a69102937d6231471b5dbb6204fe5129617082792ae468d01a3f362318");
Console.WriteLine(secret);                         // never prints the bytes
var address = EvmAddress.FromPrivateKey(secret.Span);
secret.Dispose();
Console.WriteLine($"{address} · disposed: {secret.IsDisposed}");
try { _ = secret.Span.Length; } catch (ObjectDisposedException) { Console.WriteLine("reading a disposed secret throws"); }

## 2. Kunci yang disimpan masuk keystore

Web3 Secret Storage v3 (scrypt plus AES-128-CTR plus MAC) adalah format yang dipakai geth dan MetaMask. Password yang salah menggagalkan MAC; tidak pernah menghasilkan kunci yang salah.

In [ ]:
byte[] demoKey = CryptoNative.Keccak256("notebook demo key — never fund it"u8);
string json = KeyStore.Encrypt(demoKey, "correct horse battery staple", kdf: KeyStoreKdf.Light);
var restored = KeyStore.Decrypt(json, "correct horse battery staple");
Console.WriteLine($"round trip ok: {restored.ContentEquals(demoKey)}");
restored.Dispose();
try { KeyStore.Decrypt(json, "wrong"); } catch (System.Security.Cryptography.CryptographicException e) { Console.WriteLine($"wrong password: {e.Message}"); }

## 3. Jumlah selalu presisi

`0.1 + 0.2 != 0.3` pada floating point biner. `Amount` menyimpan base unit berupa bilangan bulat. Ia menolak konversi apa pun yang akan kehilangan presisi. Aturan analyzer CNET003 menandai amount yang dibuat dari `double`.

In [ ]:
Console.WriteLine($"double: 0.1 + 0.2 = {0.1 + 0.2:R}");
var a = Amount.Parse("0.1", 18) + Amount.Parse("0.2", 18);
Console.WriteLine($"Amount: {a} ({a.BaseUnits} wei)");
try { Amount.Parse("1.0000001", 6); } catch (FormatException e) { Console.WriteLine($"too many decimals for USDC: {e.Message}"); }

## 4. Checksum menangkap salah ketik

Huruf besar-kecil EIP-55, Bech32 dan Base58Check mendeteksi sebagian besar kesalahan satu karakter. Validasi alamat sebelum mengirim apa pun.

In [ ]:
string good = "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAed";
string typo = "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAeD";
string btcTypo = "bc1qcr8te4kr609gcawutmrza0j4xv80jy8z306fyv";
new { EvmGood = EvmAddress.IsValid(good), EvmTypo = EvmAddress.IsValid(typo), BitcoinTypo = BitcoinAddress.IsValid(btcTypo) }

## 5. Perlindungan replay dan verifikasi

EIP-155 mengikat tanda tangan ke satu chain id, sehingga transaksi Sepolia tidak bisa diputar ulang di mainnet. Jika seseorang memberi Anda pesan bertanda tangan, pulihkan penandatangannya lalu bandingkan alamatnya. Jangan pernah percaya alamat yang diklaim pengirim.

In [ ]:
var tx = new EvmTransaction { ChainId = EvmChain.Sepolia.NumericChainId, Nonce = 0, To = good, Value = 1, MaxFeePerGas = 1, MaxPriorityFeePerGas = 1 };
var signed = tx.Sign(demoKey);
var decoded = EvmTransaction.DecodeSigned(signed.RawBytes);
Console.WriteLine($"signed for chain {decoded.Transaction.ChainId} (Sepolia), not 1 (mainnet)");

byte[] signature = EvmMessageSigner.SignPersonalMessage(demoKey, "I own this address");
string signer = EvmMessageSigner.RecoverPersonalMessageSigner("I own this address", HexUtil.Encode(signature));
Console.WriteLine($"claimed {EvmAddress.FromPrivateKey(demoKey)} · recovered {signer}");
Console.WriteLine($"tampered message recovers someone else: {EvmMessageSigner.RecoverPersonalMessageSigner("I own this addresS", HexUtil.Encode(signature))}");

## 6. Input berbahaya gagal dengan rapi

PSBT, transaksi, RLP, SCALE dan pesan Solana datang dari sumber yang tidak tepercaya. CI melakukan fuzzing pada setiap decoder; input rusak memunculkan `FormatException`, tidak pernah crash. Signer PSBT juga menolak input yang data UTXO-nya tidak cocok dengan transaksi yang dibelanjakannya.

In [ ]:
foreach (string bad in new[] { "cHNidP8BAA==", "70736274ff0100ffffffff", "deadbeef" })
    Console.WriteLine($"{bad,-24} → valid PSBT: {Psbt.TryParse(bad, out _)}");
try { EvmTransaction.DecodeSigned(Convert.FromHexString("f8ff00")); }
catch (FormatException e) { Console.WriteLine($"RLP: {e.Message}"); }

## 7. API key tetap di luar log

Ankr menaruh key di path URL. `RpcEndpoint.ToString()` menyamarkannya, dan `AddCryptoNet` menghapus logger HTTP setiap kali provider ber-key dikonfigurasi.

In [ ]:
var endpoint = RpcProviders.Ankr.GetEndpoint(EvmChain.Sepolia, "my-secret-api-key")!;
$"logged as: {endpoint}"

## Daftar periksa

- Kembangkan di testnet. CNET004 menandai client mainnet di build Debug.
- Simpan frasa dan kunci di user secrets, KMS atau hardware signer. CNET001 menandai yang ditulis langsung di kode.
- Simulasikan sebelum menandatangani: `PrepareTransactionAsync`, `SimulateTransactionAsync` dan `SimulateAsync` semuanya melakukannya.
- Teruskan `CancellationToken`, agar panggilan RPC yang macet tidak menggantung layanan Anda (CNET005).
- Lihat [panduan keamanan](../../docs/id/keamanan.md).